# Spatial Memory Demo: Ice & Water Navigation

**FLUX Phase 9 ARC** — Testing the dual-field spatial navigation system for ARC-AGI-3.

This notebook demonstrates:
1. **Exploration Mass Field** — tracks where agent has been
2. **Curiosity Ice Field** — highlights interesting locations
3. **Change Detection** — notices when visited areas change
4. **Path Planning** — efficient navigation via mass gradient

We'll test this with a simple ARC-AGI-3 style exploration game.

## Cell 1: Setup & Imports

In [ ]:
# Setup paths and imports
import sys
from pathlib import Path

# Add project paths
ROOT = Path('.').resolve()
if 'flux' not in str(ROOT):
    ROOT = Path('/Users/admin/Desktop/flux')

PHASES_DIR = ROOT / 'phases'
for p in [ROOT, PHASES_DIR / 'phase9_arc', PHASES_DIR / 'phase8_8']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import torch
import random
import time
from IPython.display import clear_output, display, HTML
import matplotlib.pyplot as plt
import numpy as np

# Import our spatial memory module
from spatial_memory import SpatialMemory, SpatialARCAgent, NavigationTarget

print("✓ Imports successful")
print(f"  PyTorch: {torch.__version__}")
print(f"  Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Cell 2: Define ARC-AGI-3 Style Game Environment

A simple exploration game where:
- Agent must find all "treasures" (colored cells)
- Some treasures only appear after visiting certain locations
- This tests change detection + curiosity-driven navigation

In [ ]:
class TreasureHuntEnv:
    """
    ARC-AGI-3 style exploration game.
    
    Rules:
    - Grid with hidden treasures (appear as colored cells)
    - Some treasures are visible from start
    - Some treasures only appear when you visit a trigger location
    - Goal: find all treasures in minimum steps
    
    This tests:
    - Curiosity-driven exploration (go to colored cells)
    - Change detection (notice new treasures appearing)
    - Efficient navigation (don't revisit unnecessarily)
    """
    
    def __init__(self, size: int = 10, num_treasures: int = 4, num_hidden: int = 2):
        self.size = size
        self.num_treasures = num_treasures
        self.num_hidden = num_hidden
        self.reset()
    
    def reset(self):
        """Reset the environment."""
        self.agent_pos = [0, 0]
        self.steps = 0
        self.found_treasures = set()
        
        # Place visible treasures (colors 1-4)
        self.visible_treasures = {}
        positions = random.sample(
            [(r, c) for r in range(self.size) for c in range(self.size) if (r, c) != (0, 0)],
            k=self.num_treasures
        )
        for i, pos in enumerate(positions):
            self.visible_treasures[pos] = i + 1  # Colors 1-4
        
        # Place hidden treasures (appear after trigger)
        # Trigger: visiting any visible treasure reveals ONE hidden treasure
        self.hidden_treasures = {}  # (trigger_pos) -> (treasure_pos, color)
        self.revealed_hidden = set()
        
        trigger_positions = list(self.visible_treasures.keys())[:self.num_hidden]
        for i, trigger in enumerate(trigger_positions):
            # Place hidden treasure somewhere else
            available = [(r, c) for r in range(self.size) for c in range(self.size) 
                        if (r, c) not in self.visible_treasures and (r, c) != (0, 0)]
            if available:
                hidden_pos = random.choice(available)
                self.hidden_treasures[trigger] = (hidden_pos, 5 + i)  # Colors 5-6
        
        return self.get_observation()
    
    def get_observation(self):
        """Get current grid state."""
        grid = [[0] * self.size for _ in range(self.size)]
        
        # Place visible treasures
        for pos, color in self.visible_treasures.items():
            if pos not in self.found_treasures:
                grid[pos[0]][pos[1]] = color
        
        # Place revealed hidden treasures
        for trigger, (pos, color) in self.hidden_treasures.items():
            if trigger in self.revealed_hidden and pos not in self.found_treasures:
                grid[pos[0]][pos[1]] = color
        
        # Agent marker (color 9)
        grid[self.agent_pos[0]][self.agent_pos[1]] = 9
        
        return grid
    
    def step(self, action: int):
        """
        Take action and return new state.
        
        Actions: 0=up, 1=down, 2=left, 3=right
        """
        # Move agent
        dr, dc = [(-1, 0), (1, 0), (0, -1), (0, 1)][action]
        new_r = max(0, min(self.size - 1, self.agent_pos[0] + dr))
        new_c = max(0, min(self.size - 1, self.agent_pos[1] + dc))
        self.agent_pos = [new_r, new_c]
        self.steps += 1
        
        pos = tuple(self.agent_pos)
        
        # Check if found treasure
        treasure_found = False
        if pos in self.visible_treasures and pos not in self.found_treasures:
            self.found_treasures.add(pos)
            treasure_found = True
            
            # Reveal hidden treasure if this was a trigger
            if pos in self.hidden_treasures:
                self.revealed_hidden.add(pos)
        
        # Check for revealed hidden treasures
        for trigger, (hidden_pos, color) in self.hidden_treasures.items():
            if trigger in self.revealed_hidden and pos == hidden_pos and pos not in self.found_treasures:
                self.found_treasures.add(pos)
                treasure_found = True
        
        # Check if done
        total_treasures = len(self.visible_treasures) + len(self.hidden_treasures)
        done = len(self.found_treasures) >= total_treasures
        
        return self.get_observation(), treasure_found, done
    
    def get_stats(self):
        """Get game statistics."""
        total = len(self.visible_treasures) + len(self.hidden_treasures)
        return {
            'steps': self.steps,
            'found': len(self.found_treasures),
            'total': total,
            'revealed_hidden': len(self.revealed_hidden),
            'efficiency': len(self.found_treasures) / max(1, self.steps) if self.steps > 0 else 0,
        }

print("✓ TreasureHuntEnv defined")

## Cell 3: Visualization Helpers

In [ ]:
# Color mapping for visualization
COLORS = {
    0: '#1a1a2e',  # Black (empty)
    1: '#e63946',  # Red (treasure 1)
    2: '#2a9d8f',  # Teal (treasure 2)
    3: '#e9c46a',  # Yellow (treasure 3)
    4: '#9b59b6',  # Purple (treasure 4)
    5: '#00ff00',  # Green (hidden 1)
    6: '#00ffff',  # Cyan (hidden 2)
    7: '#ff8800',  # Orange
    8: '#ff0088',  # Pink
    9: '#ffffff',  # White (agent)
}

def plot_game_state(env, agent, title="Game State", show_fields=True):
    """Plot the game state with optional field overlay."""
    
    if show_fields:
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    else:
        fig, axes = plt.subplots(1, 1, figsize=(6, 6))
        axes = [axes]
    
    # Plot 1: Game grid
    grid = env.get_observation()
    rgb_grid = np.zeros((env.size, env.size, 3))
    
    for r in range(env.size):
        for c in range(env.size):
            color = COLORS.get(grid[r][c], '#888888')
            rgb_grid[r, c] = [int(color[i:i+2], 16)/255 for i in (1, 3, 5)]
    
    axes[0].imshow(rgb_grid)
    axes[0].set_title(f"{title}\nStep: {env.steps}, Found: {len(env.found_treasures)}")
    axes[0].set_xticks(range(env.size))
    axes[0].set_yticks(range(env.size))
    axes[0].grid(True, color='gray', linewidth=0.5)
    
    if show_fields and len(axes) > 1:
        # Plot 2: Exploration Mass
        mass = agent.spatial_memory.exploration_mass[:env.size, :env.size].cpu().numpy()
        im = axes[1].imshow(mass, cmap='hot', vmin=0)
        axes[1].set_title("Exploration Mass\n(Where I've been)")
        plt.colorbar(im, ax=axes[1])
        
        # Plot 3: Curiosity Ice
        ice = agent.spatial_memory.curiosity_field[:env.size, :env.size].cpu().numpy()
        im = axes[2].imshow(ice, cmap='cool', vmin=0)
        axes[2].set_title("Curiosity Ice\n(What's interesting)")
        plt.colorbar(im, ax=axes[2])
    
    plt.tight_layout()
    plt.show()

def print_ascii_state(env, agent):
    """Print ASCII visualization."""
    print(agent.get_visualization((env.size, env.size)))
    stats = agent.get_stats((env.size, env.size))
    game_stats = env.get_stats()
    print(f"\nAgent: coverage={stats['coverage']:.1%}, ice_peaks={stats['ice_peaks']}")
    print(f"Game: found={game_stats['found']}/{game_stats['total']}, steps={game_stats['steps']}")

print("✓ Visualization helpers defined")

## Cell 4: Test Basic Spatial Memory Operations

In [ ]:
# Create spatial memory and test basic operations
memory = SpatialMemory(max_size=10, device='cpu')
memory.reset()

# Create test grid with some objects
test_grid = [
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 1, 0, 0, 0, 0, 0, 0],  # Red block
    [0, 0, 1, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 2, 0, 0],  # Single green cell (isolated → high curiosity)
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 3, 3],  # Corner object
]

# Observe from position (0, 0)
result = memory.observe((0, 0), test_grid)
print(f"\n1. Observed from (0,0):")
print(f"   Change detected: {result['change_detected']}")

# Detect anomalies
anomalies = memory.detect_anomalies(test_grid, top_k=5)
print(f"\n2. Top anomalies (ice locations):")
for pos, score, reason in anomalies:
    print(f"   {pos}: score={score:.2f}, reason={reason}")

# Get navigation target
target = memory.get_navigation_target((0, 0), (10, 10))
if target:
    print(f"\n3. Navigation target:")
    print(f"   Position: {target.position}")
    print(f"   Curiosity: {target.curiosity_score:.2f}")
    print(f"   Reason: {target.reason}")
    print(f"   Path: {target.path[:5]}..." if len(target.path) > 5 else f"   Path: {target.path}")

# Show stats
stats = memory.get_stats((10, 10))
print(f"\n4. Memory stats:")
for k, v in stats.items():
    print(f"   {k}: {v}")

## Cell 5: Run the Treasure Hunt Game with Spatial Agent

In [ ]:
# Create game and agent
env = TreasureHuntEnv(size=10, num_treasures=4, num_hidden=2)
agent = SpatialARCAgent(max_size=10, device='cpu')

# Reset both
grid = env.reset()
agent.reset(initial_pos=(0, 0))

print("Game Setup")
print("="*50)
print(f"Grid size: {env.size}x{env.size}")
print(f"Visible treasures: {len(env.visible_treasures)}")
print(f"Hidden treasures: {len(env.hidden_treasures)}")
print(f"Total to find: {len(env.visible_treasures) + len(env.hidden_treasures)}")
print()
print("Treasure locations (colors 1-4 visible, 5-6 hidden):")
for pos, color in env.visible_treasures.items():
    hidden_note = f" → reveals hidden at {env.hidden_treasures[pos][0]}" if pos in env.hidden_treasures else ""
    print(f"  {pos}: color {color}{hidden_note}")

# Show initial state
plot_game_state(env, agent, "Initial State")

## Cell 6: Run Game Loop with Visualization

In [ ]:
# Run the game
MAX_STEPS = 100
PRINT_EVERY = 10

print("Running Treasure Hunt...")
print("="*50)

done = False
step = 0
events = []

while not done and step < MAX_STEPS:
    # Agent decides action based on spatial memory
    pos = tuple(env.agent_pos)
    action = agent.observe_and_decide(grid, pos)
    
    # Take step in environment
    grid, treasure_found, done = env.step(action)
    step += 1
    
    # Log events
    if treasure_found:
        events.append(f"Step {step}: Found treasure at {pos}!")
    
    # Check for newly revealed hidden treasures (change detection!)
    if len(env.revealed_hidden) > 0:
        for trigger in env.revealed_hidden:
            if trigger in env.hidden_treasures:
                hidden_pos, color = env.hidden_treasures[trigger]
                if f"revealed {hidden_pos}" not in str(events):
                    events.append(f"Step {step}: Hidden treasure revealed at {hidden_pos}! (triggered by {trigger})")
    
    # Periodic update
    if step % PRINT_EVERY == 0:
        stats = env.get_stats()
        agent_stats = agent.get_stats((env.size, env.size))
        print(f"Step {step}: found={stats['found']}/{stats['total']}, " +
              f"coverage={agent_stats['coverage']:.1%}, " +
              f"ice_peaks={agent_stats['ice_peaks']}")

# Final results
print()
print("="*50)
print("GAME OVER")
print("="*50)

stats = env.get_stats()
print(f"Steps taken: {stats['steps']}")
print(f"Treasures found: {stats['found']}/{stats['total']}")
print(f"Hidden revealed: {stats['revealed_hidden']}/{len(env.hidden_treasures)}")
print(f"Success: {'✓ YES' if done else '✗ NO'}")

print("\nKey events:")
for event in events:
    print(f"  {event}")

## Cell 7: Visualize Final State

In [ ]:
# Show final state with all three views
plot_game_state(env, agent, "Final State", show_fields=True)

# ASCII visualization
print("\nASCII Spatial Memory View:")
print_ascii_state(env, agent)

## Cell 8: Test Change Detection

The key feature: when the agent revisits a location and something has changed, **ice forms** (curiosity spikes).

In [ ]:
# Test change detection explicitly
print("Change Detection Test")
print("="*50)

# Create fresh memory
mem = SpatialMemory(max_size=5, device='cpu')
mem.reset()

# Grid state 1: empty
grid1 = [[0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0]]

# First observation
result1 = mem.observe((2, 2), grid1)
print(f"1. First observation at (2,2):")
print(f"   Change detected: {result1['change_detected']}")
print(f"   Ice: {mem.curiosity_field.sum().item():.2f}")

# Grid state 2: something appeared!
grid2 = [[0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0],
         [0, 0, 1, 0, 0],  # Changed! Color 1 appeared
         [0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0]]

# Second observation - should detect change!
result2 = mem.observe((2, 2), grid2)
print(f"\n2. Second observation (something appeared):")
print(f"   Change detected: {result2['change_detected']}")
print(f"   New ice positions: {result2['new_ice_positions']}")
print(f"   Curiosity delta: {result2['curiosity_delta']:.2f}")
print(f"   Total ice: {mem.curiosity_field.sum().item():.2f}")

# Visualization
print("\n3. Spatial memory visualization:")
print(mem.visualize((5, 5), (2, 2)))

## Cell 9: Compare Agent Performance

Compare our Spatial Memory agent against a random agent.

In [ ]:
def run_game(agent_type='spatial', size=10, seed=42):
    """Run a single game and return stats."""
    random.seed(seed)
    torch.manual_seed(seed)
    
    env = TreasureHuntEnv(size=size, num_treasures=4, num_hidden=2)
    grid = env.reset()
    
    if agent_type == 'spatial':
        agent = SpatialARCAgent(max_size=size, device='cpu')
        agent.reset((0, 0))
    
    done = False
    max_steps = 200
    
    while not done and env.steps < max_steps:
        pos = tuple(env.agent_pos)
        
        if agent_type == 'spatial':
            action = agent.observe_and_decide(grid, pos)
        else:  # random
            action = random.randint(0, 3)
        
        grid, _, done = env.step(action)
    
    return env.get_stats()

# Run comparison
print("Agent Performance Comparison")
print("="*50)

NUM_GAMES = 10
spatial_results = []
random_results = []

for i in range(NUM_GAMES):
    spatial_results.append(run_game('spatial', seed=i))
    random_results.append(run_game('random', seed=i))

def avg_stats(results):
    return {
        'avg_steps': sum(r['steps'] for r in results) / len(results),
        'avg_found': sum(r['found'] for r in results) / len(results),
        'completion_rate': sum(1 for r in results if r['found'] == r['total']) / len(results),
    }

spatial_avg = avg_stats(spatial_results)
random_avg = avg_stats(random_results)

print(f"\nSpatial Memory Agent ({NUM_GAMES} games):")
print(f"  Average steps: {spatial_avg['avg_steps']:.1f}")
print(f"  Average treasures found: {spatial_avg['avg_found']:.1f}/6")
print(f"  Completion rate: {spatial_avg['completion_rate']:.1%}")

print(f"\nRandom Agent ({NUM_GAMES} games):")
print(f"  Average steps: {random_avg['avg_steps']:.1f}")
print(f"  Average treasures found: {random_avg['avg_found']:.1f}/6")
print(f"  Completion rate: {random_avg['completion_rate']:.1%}")

print(f"\nSpatial Memory Advantage:")
print(f"  Steps saved: {random_avg['avg_steps'] - spatial_avg['avg_steps']:.1f}")
print(f"  Better completion: +{(spatial_avg['completion_rate'] - random_avg['completion_rate'])*100:.0f}%")

## Cell 10: Summary

The **Ice & Water** spatial memory model provides:

| Feature | How It Works | ARC-AGI-3 Benefit |
|---------|--------------|-------------------|
| **Exploration Mass** | Track where we've been | Efficient path planning |
| **Curiosity Ice** | Highlight anomalies | Go to interesting things |
| **Change Detection** | Notice when visited areas change | React to dynamic environments |
| **Mass Gradient Navigation** | A* with known-territory bonus | Don't waste steps |

This gives FLUX agents **physics-native spatial awareness** — unlike LLMs that treat space as text tokens.

In [ ]:
print("✓ Spatial Memory Demo Complete")
print()
print("Key files created:")
print("  • phases/phase9_arc/spatial_memory.py")
print("  • phases/phase9_arc/SPATIAL_MEMORY_SPEC.md")
print("  • notebooks/spatial_memory_demo.ipynb")
print()
print("Next steps:")
print("  1. Integrate with arc_agent.py")
print("  2. Test on real ARC-AGI-3 environments")
print("  3. Tune curiosity thresholds")
print("  4. Add causal tracking (why ice formed)")